In [ ]:
import mercury as mr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df_nodos      = pd.read_csv('data/clean/nodos_autores.csv')
df_aristas    = pd.read_csv('data/clean/aristas_colaboraciones.csv')
df_pos        = pd.read_csv('data/clean/posiciones.csv')
df_central    = pd.read_csv('data/clean/centralidad.csv')
df_nodos_sub  = pd.read_csv('data/clean/nodos_sub.csv')
df_aristas_sub= pd.read_csv('data/clean/aristas_sub.csv')

print('Datos cargados OK')

In [ ]:
top_inst = df_nodos_sub['institucion'].value_counts().head(15).index.tolist()

inst_sel = mr.MultiSelect(
    value=top_inst[:5],
    choices=top_inst,
    label='Instituciones a destacar'
)

anio_corte = mr.Slider(
    value=2024, min=2018, max=2024, step=1,
    label='Año de corte'
)

## La ciencia no se hace sola

Cada paper científico es el resultado de una conversación. Cuando esas conversaciones
se acumulan durante años, forman una red invisible que define qué tan conectada
y potente es la comunidad científica de un país.

Este artículo explora esa red en México entre 2018 y 2024 usando datos de
**OpenAlex**, el catálogo académico de acceso libre más grande del mundo.
**¿Quién conecta a quién?**

---
## Visualización 1: El mapa de la ciencia mexicana

Cada punto es un investigador. Cada línea es una colaboración publicada.
Los puntos **azules** son las instituciones seleccionadas.
Verás clusters bien definidos — la UNAM, el CINVESTAV y el IPN forman
galaxias propias: conectadas, pero no fusionadas.

*Pasa el cursor sobre cada punto para ver nombre e institución.*

In [ ]:
df_plot = df_nodos_sub.merge(df_pos, on='autor_id')
df_e = df_aristas_sub.merge(
    df_pos.rename(columns={'autor_id':'source','x':'x0','y':'y0'}), on='source')
df_e = df_e.merge(
    df_pos.rename(columns={'autor_id':'target','x':'x1','y':'y1'}), on='target')

edge_x, edge_y = [], []
for _, r in df_e.iterrows():
    edge_x += [r.x0, r.x1, None]
    edge_y += [r.y0, r.y1, None]

colores = ['royalblue' if i in inst_sel.value else '#cccccc'
           for i in df_plot['institucion']]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.4, color='#dddddd'), hoverinfo='none', showlegend=False))
fig1.add_trace(go.Scatter(
    x=df_plot['x'], y=df_plot['y'], mode='markers',
    marker=dict(
        size=[max(4, min(16, a**0.6)) for a in df_plot['articulos']],
        color=colores, opacity=0.85, line=dict(width=0.5, color='white')
    ),
    text=['<b>'+n+'</b><br>'+i+'<br>'+str(a)+' artículos'
          for n,i,a in zip(df_plot['nombre'],df_plot['institucion'],df_plot['articulos'])],
    hovertemplate='%{text}<extra></extra>', showlegend=False
))
fig1.update_layout(
    title='Red de coautoría científica en México (top 400 investigadores)',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=600, paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(l=20,r=20,t=50,b=20)
)
fig1.show()

---
## Visualización 2: Los investigadores que mueven los hilos

La **centralidad de intermediación** mide cuántas veces un investigador aparece
en el camino más corto entre dos colegas que sin él no estarían conectados.
No es el más prolífico ni el más citado — es el **puente**.

*Tamaño = artículos publicados. Color = institución.*

In [ ]:
fig2 = px.scatter(
    df_central.head(150),
    x='betweenness', y='citas',
    size='articulos', color='institucion',
    hover_name='nombre',
    labels={'betweenness':'Centralidad de intermediación','citas':'Citas acumuladas','articulos':'Artículos'},
    title='Conectores vs. Impacto — Top 150 investigadores',
    height=500, template='plotly_white', size_max=28
)
for _, row in df_central.head(5).iterrows():
    fig2.add_annotation(x=row['betweenness'], y=row['citas'],
        text=row['nombre'].split()[-1], showarrow=True, arrowhead=2, font=dict(size=10))
fig2.show()

---
## Visualización 3: ¿Cómo ha crecido la red?

Las barras muestran nuevas colaboraciones por año.
La línea roja muestra el promedio de citas de esos artículos.
¿Más colaboraciones = más impacto? Los datos tienen la respuesta.

In [ ]:
evolucion = (
    df_aristas[df_aristas['primera_colab'] <= anio_corte.value]
    .groupby('primera_colab')
    .agg(nuevas_colabs=('peso','count'), citas_promedio=('citas_max','mean'))
    .reset_index().rename(columns={'primera_colab':'anio'})
)
fig3 = go.Figure()
fig3.add_trace(go.Bar(x=evolucion['anio'], y=evolucion['nuevas_colabs'],
    name='Nuevas colaboraciones', marker_color='steelblue', opacity=0.75, yaxis='y1'))
fig3.add_trace(go.Scatter(x=evolucion['anio'], y=evolucion['citas_promedio'],
    name='Citas promedio', mode='lines+markers',
    line=dict(color='crimson', width=2.5), marker=dict(size=8), yaxis='y2'))
fig3.update_layout(
    title='Evolución de la red científica mexicana',
    xaxis=dict(title='Año', tickmode='linear', dtick=1),
    yaxis=dict(title='Nuevas colaboraciones', side='left'),
    yaxis2=dict(title='Citas promedio', overlaying='y', side='right', showgrid=False),
    legend=dict(x=0.05, y=0.95), height=450, template='plotly_white', bargap=0.3
)
fig3.show()

---
## ¿Qué encontramos?

La red científica mexicana es **densa en el centro, dispersa en los bordes**.
Un núcleo de investigadores de la UNAM, CINVESTAV e IPN concentra la mayoría
de conexiones. Los puentes entre instituciones son pocos pero críticos.

La pregunta que queda abierta: **¿cómo se diseñan políticas que incentiven
las conexiones correctas?**

---
*Datos: OpenAlex API · CC0 · Python / NetworkX / Plotly*